# 1. Data and splits

Loads TORGO, checks it, and confirms that the splits saved in the repo rebuild exactly. No GPU needed. Takes about 3 minutes.

In [ ]:
# Setup: run this cell first. On Colab it clones the repo, installs packages and mounts Google Drive.
REPO_URL = "https://github.com/armaan-khoja/clear-voice"
SMOKE = False  # True = a quick check on a handful of clips (minutes, not hours)

import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/clear-voice"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/clear-voice"], check=True)
    os.chdir("/content/clear-voice")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    from google.colab import userdata
    WORK = os.environ.get("CLEARVOICE_WORK")  # a local folder instead of Google Drive, if set
    if not WORK:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK = "/content/drive/MyDrive/clearvoice"
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN secret: downloads still work, uploading models will not.")
else:
    WORK = os.path.abspath("work")
if SMOKE:
    WORK = f"{WORK}/smoke"  # quick checks never overwrite real models or results
sys.path.insert(0, os.getcwd())
os.makedirs(f"{WORK}/results", exist_ok=True)
os.makedirs(f"{WORK}/models", exist_ok=True)
SPLITS = "splits"
LIMIT = 6 if SMOKE else None
print("Results and models go to", WORK)

In [ ]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "No GPU: Runtime > Change runtime type > L4 GPU, then rerun the setup cell.")

In [ ]:
from clearvoice.pipeline import load_torgo
parquets, meta = load_torgo(WORK)  # downloads TORGO (about 1.6 GB) the first time in each session
print(len(meta), "clips from", meta.speaker.nunique(), "speakers")

**Minutes of speech per speaker** (both microphone copies counted).

In [ ]:
meta.groupby(["speaker", "severity"]).duration.sum().div(60).round(1)

**Data checks.** Every utterance was recorded on two microphones; both copies must say the same thing.

In [ ]:
from clearvoice.data import check_mic_pairs
print("Utterances whose two microphone copies disagree:", check_mic_pairs(meta).utt_key.nunique())
print("Clips longer than 30 s (dropped from every split):", (meta.duration > 30).sum())

**Rebuild the splits** and check they match the CSVs in `splits/`. Test and validation sentences never appear in training; held-out speakers never appear in training.

In [ ]:
from clearvoice.splits import assign_prompt_splits, fold_splits, read_split
ps = assign_prompt_splits(meta)
for held in ["A", "B"]:
    for part, df in fold_splits(meta, ps, held).items():
        saved = read_split(SPLITS, f"hold{held}", part)
        assert list(saved.row_id) == list(df.row_id), (held, part)
print("Splits in the repo match a fresh rebuild.")